## 面试问题

每步 action 合法性怎么用结构化输出/受控解码约束？

## 回答主线

模型提议的动作必须落在合法空间：allowlist 动作名 + 参数 schema + 前置条件，由确定性外壳校验，模型只提议。本 Notebook 用退款 agent 的合法动作 `{lookup, refund, finish}`（refund 需 amount），对比无校验（执行未知动作 cancel_all、执行缺 amount 的退款）与有校验（拒绝非法动作并兜底追问）。

## 真实案例

合法动作 `{lookup, refund, finish}`，`refund` 必须带 `amount`。模型提议序列里混入非法的 `cancel_all` 和缺 `amount` 的 `refund`。数据为教学动作，不代表真实工具集。

In [1]:
ACTION_SCHEMA = {  # 定义合法动作空间。
    "lookup": [],  # lookup 无必填参数。
    "refund": ["amount"],  # refund 必须带 amount 参数。
    "finish": [],  # finish 无必填参数。
}  # 结束动作 schema 定义。

proposals = [  # 模型逐步提议的动作序列。
    {"name": "lookup", "params": {}},  # 合法：查询。
    {"name": "refund", "params": {}},  # 非法：refund 缺 amount。
    {"name": "cancel_all", "params": {}},  # 非法：不存在的动作。
    {"name": "refund", "params": {"amount": 88}},  # 合法：带 amount 的退款。
]  # 结束提议序列定义。

print("合法动作:", list(ACTION_SCHEMA.keys()))  # 展示合法动作空间。
for p in proposals:  # 逐条打印模型提议。
    print("  提议:", p["name"], p["params"])  # 展示每个提议。

合法动作: ['lookup', 'refund', 'finish']
  提议: lookup {}
  提议: refund {}
  提议: cancel_all {}
  提议: refund {'amount': 88}


## 基线（Baseline）

反面基线：无校验直接执行模型提议。结果缺 `amount` 的退款带着 `None` 执行、不存在的 `cancel_all` 也被执行（越权）。

In [2]:
def execute_unchecked(proposal):  # 无校验：直接执行模型提议。
    name = proposal["name"]  # 取动作名。
    if name == "refund":  # 退款动作。
        amount = proposal["params"].get("amount")  # 可能取到 None。
        return {"executed": "refund", "amount": amount}  # 带着可能为 None 的金额执行。
    if name == "lookup":  # 查询动作。
        return {"executed": "lookup"}  # 执行查询。
    return {"executed": name, "warning": "unknown_action"}  # 未知动作也被执行。

unchecked_results = [execute_unchecked(p) for p in proposals]  # 无校验执行全部提议。
for r in unchecked_results:  # 逐条打印执行结果。
    print("无校验执行:", r)  # 展示非法动作和缺参也被执行。

无校验执行: {'executed': 'lookup'}
无校验执行: {'executed': 'refund', 'amount': None}
无校验执行: {'executed': 'cancel_all', 'warning': 'unknown_action'}
无校验执行: {'executed': 'refund', 'amount': 88}


## 失败案例与修正

无校验会执行越权动作和缺参退款。修正是校验器：allowlist 校验动作名、schema 校验必填参数，非法动作拒绝并兜底为追问。

In [3]:
def validate_action(proposal, schema):  # 校验动作合法性：allowlist 加必填参数。
    name = proposal["name"]  # 取动作名。
    if name not in schema:  # 动作名不在 allowlist。
        return False, "unknown_action"  # 拒绝未知动作。
    required = schema[name]  # 取该动作的必填参数。
    for key in required:  # 逐个检查必填参数。
        if key not in proposal["params"]:  # 缺少必填参数。
            return False, "missing_param:" + key  # 拒绝缺参动作。
    return True, "ok"  # 通过校验。

In [4]:
def execute_checked(proposal, schema):  # 校验后执行：非法则兜底为追问。
    ok, reason = validate_action(proposal, schema)  # 先校验合法性。
    if not ok:  # 非法动作。
        return {"executed": "ask_user", "rejected": proposal["name"], "reason": reason}  # 兜底为追问并记录原因。
    return {"executed": proposal["name"], "params": proposal["params"]}  # 合法则执行。

checked_results = [execute_checked(p, ACTION_SCHEMA) for p in proposals]  # 校验后执行全部提议。
for r in checked_results:  # 逐条打印结果。
    print("校验执行:", r)  # 展示非法动作被拒绝兜底。

校验执行: {'executed': 'lookup', 'params': {}}
校验执行: {'executed': 'ask_user', 'rejected': 'refund', 'reason': 'missing_param:amount'}
校验执行: {'executed': 'ask_user', 'rejected': 'cancel_all', 'reason': 'unknown_action'}
校验执行: {'executed': 'refund', 'params': {'amount': 88}}


In [5]:
rejected = [r for r in checked_results if "rejected" in r]  # 收集被拒绝的动作。
unchecked_bad = sum(1 for r in unchecked_results if r["executed"] == "refund" and r.get("amount") is None)  # 无校验缺参退款数。
unchecked_unknown = sum(1 for r in unchecked_results if r.get("warning") == "unknown_action")  # 无校验越权动作数。
print("无校验: 缺参退款数", unchecked_bad, "越权未知动作数", unchecked_unknown)  # 展示无校验的危险。
print("有校验: 被拒绝非法动作数", len(rejected))  # 展示校验拦截了非法动作。

无校验: 缺参退款数 1 越权未知动作数 1
有校验: 被拒绝非法动作数 2


## 结果解读

无校验执行了 1 次缺 `amount` 的退款和 1 次越权 `cancel_all`；有校验拒绝了这 2 个非法动作、兜底为追问，只放行 2 个合法动作。要点：allowlist 优于 blocklist，参数 schema 校验必填，非法要有兜底路径而非崩溃。

In [6]:
assert unchecked_bad == 1  # 无校验执行了一次缺 amount 的退款。
assert unchecked_unknown == 1  # 无校验执行了一次未知动作。
assert len(rejected) == 2  # 校验拒绝了两个非法动作。
assert checked_results[0]["executed"] == "lookup"  # 合法 lookup 正常执行。
assert checked_results[3]["executed"] == "refund"  # 合法 refund 正常执行。
assert checked_results[1]["reason"] == "missing_param:amount"  # 缺参退款被正确拒绝。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
